### Partie 1 : Ingestion et nettoyage avec Spark (4-5h)

**Competence evaluee : C2.1 - Collecter des donnees en respectant les normes et standards**

#### Etape 1.1 : Exploration initiale

In [1]:
#CONSTANTES

import os


DATA_DIR = "../data"
DATA_DIR = os.path.join(DATA_DIR, "..", "data")
OUTPUT_DIR = os.path.join(DATA_DIR, "..", "output", "consommation_clean")

CONSOMMATION_PATH = os.path.join(DATA_DIR, "consommations_raw.csv")
BATIMENTS_PATH = os.path.join(DATA_DIR, "batiments.csv")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


spark = SparkSession.builder \
    .appName("ECF2 - Exploration") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

#moins de logs
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

print("\n[1/6] Chargement des donnees brutes...")
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(CONSOMMATION_PATH)

print(f"Nombre de lignes: {df_raw.count():,}")
print(f"Nombre de colonnes: {len(df_raw.columns)}")

#schema
print(df_raw.printSchema())

df_raw.show()

Spark version: 3.5.7
Spark UI: http://host.docker.internal:4041

[1/6] Chargement des donnees brutes...
Nombre de lignes: 7,758,868
Nombre de colonnes: 5
root
 |-- batiment_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- type_energie: string (nullable = true)
 |-- consommation: string (nullable = true)
 |-- unite: string (nullable = true)

None
+-----------+-------------------+------------+------------+-----+
|batiment_id|          timestamp|type_energie|consommation|unite|
+-----------+-------------------+------------+------------+-----+
|    BAT0141|2023-12-21 13:00:00|         gaz|      342.34|  kWh|
|    BAT0080|   08/08/2023 13:00|         gaz|     1256.73|  kWh|
|    BAT0122|06/13/2024 11:00:00|         eau|      133.57|   m3|
|    BAT0033|2023-06-25 00:00:00|         eau|        0.23|   m3|
|    BAT0064|11/29/2024 04:00:00|         gaz|       12.26|  kWh|
|    BAT0052|09/15/2024 18:00:00|         gaz|      701.01|  kWh|
|    BAT0097|2024-01-25 12:00:00

### Problèmes de typage : 
- batiment_id est nullable
- les timestamp sont des strings avec une syntaxe créative
- consommation est de type string au lieu de float, avec donc des variations xx.xx , xx,xx , "xx.xx" ou des valeurs non numériques null, "null", --- etc
- le champ consommation représente des données différentes selon lignes (eau, électricité, gaz)

In [ ]:
df_non_numeric = df_raw.filter(
    ~F.col("consommation").rlike("^-?[0-9]+[.,]?[0-9]*$")
)
print(f"Nombre de valeurs non numeriques: {df_non_numeric.count():,}")
df_non_numeric.select("consommation").distinct().show()

df_with_comma = df_raw.filter(F.col("consommation").contains(","))
print(f"Nombre de valeurs avec virgule: {df_with_comma.count():,}")
df_with_comma.select("consommation").show(5)



Nombre de valeurs non numeriques: 38,975
+------------+
|consommation|
+------------+
|        null|
|         N/A|
|      erreur|
|         ---|
+------------+

Nombre de valeurs avec virgule: 925,392
+------------+
|consommation|
+------------+
|       10,10|
|       72,29|
|       17,82|
|        0,92|
|      290,65|
+------------+
only showing top 5 rows

Exemples de formats de timestamp:
+-------------------+
|timestamp          |
+-------------------+
|2023-12-21 13:00:00|
|11/29/2024 04:00:00|
|09/15/2024 18:00:00|
|20/12/2023 07:00   |
|29/04/2024 13:00   |
|2024-06-15T14:00:00|
|22/03/2024 11:00   |
|2023-04-07 13:00:00|
|2024-03-11T11:00:00|
|08/21/2024 17:00:00|
|2024-05-27 04:00:00|
|08/02/2023 16:00   |
|2024-11-23 17:00:00|
|2023-07-06T05:00:00|
|2024-04-30 02:00:00|
|08/12/2023 06:00   |
|16/11/2024 21:00   |
|12/08/2024 04:00:00|
|2024-05-27 10:00:00|
|11/03/2023 09:00:00|
+-------------------+
only showing top 20 rows



In [ ]:
print("Exemples de formats de timestamp:")
df_raw.select("timestamp").distinct().show(20, truncate=False)

Exemples de formats de timestamp:
+-------------------+
|timestamp          |
+-------------------+
|2023-12-21 13:00:00|
|11/29/2024 04:00:00|
|09/15/2024 18:00:00|
|20/12/2023 07:00   |
|29/04/2024 13:00   |
|2024-06-15T14:00:00|
|22/03/2024 11:00   |
|2023-04-07 13:00:00|
|2024-03-11T11:00:00|
|08/21/2024 17:00:00|
|2024-05-27 04:00:00|
|08/02/2023 16:00   |
|2024-11-23 17:00:00|
|2023-07-06T05:00:00|
|2024-04-30 02:00:00|
|08/12/2023 06:00   |
|16/11/2024 21:00   |
|12/08/2024 04:00:00|
|2024-05-27 10:00:00|
|11/03/2023 09:00:00|
+-------------------+
only showing top 20 rows



### Stats descriptives par consommation

In [11]:
# Convertir consommation en double
df_conso_numeric = df_raw.withColumn(
    "conso_clean",
    F.regexp_replace(F.col("consommation"), ",", ".").cast("double")
)

# ignorer conso nulles
stats_by_type_energy = df_conso_numeric.filter(F.col("conso_clean").isNotNull()) \
    .groupBy("type_energie") \
    .agg(
        F.count("*").alias("count"),
        F.round(F.mean("conso_clean"), 2).alias("mean"),
        F.round(F.stddev("conso_clean"), 2).alias("stddev"),
        F.round(F.min("conso_clean"), 2).alias("min"),
        F.round(F.max("conso_clean"), 2).alias("max"),
        F.round(F.expr("percentile(conso_clean, 0.5)"), 2).alias("median")
    ) \
    .orderBy("type_energie")

print("Statistiques par type d'énergie:")
stats_by_type_energy.show()

Statistiques par type d'énergie:
+------------+-------+------+-------+--------+--------+------+
|type_energie|  count|  mean| stddev|     min|     max|median|
+------------+-------+------+-------+--------+--------+------+
|         eau|2573156|204.36|2398.57| -657.01|49999.23|  7.52|
| electricite|2573364|430.64|2429.51|-4003.35|49999.13|108.56|
|         gaz|2573373|560.94|2465.81|-5963.49|49999.49|160.57|
+------------+-------+------+-------+--------+--------+------+



Présence de  :
- valeurs négatives
- valeurs suspectes (> 49000 malgré une median qui ne dépasse pas 160 pour la plus haute)

### Batiments les plus mesurés

In [15]:
df_bat_plus_mesures = df_conso_numeric.groupBy("batiment_id").count().orderBy("count", ascending= False).limit(10)

df_bat_plus_mesures.show()

+-----------+-----+
|batiment_id|count|
+-----------+-----+
|    BAT0086|53275|
|    BAT0002|53257|
|    BAT0145|53255|
|    BAT0117|53254|
|    BAT0047|53254|
|    BAT0051|53246|
|    BAT0093|53242|
|    BAT0078|53235|
|    BAT0146|53235|
|    BAT0046|53233|
+-----------+-----+



### Rapport de qualité des données

In [ ]:
# nulls
null_counts = df_raw.select([
    F.count(F.when(F.col(c).isNull() | (F.col(c) == ""), c)).alias(c)
    for c in df_raw.columns
])

print("Nombre de valeurs nulles/vides par colonne:")
null_counts.show()

Nombre de valeurs nulles/vides par colonne:
+-----------+---------+------------+------------+-----+
|batiment_id|timestamp|type_energie|consommation|unite|
+-----------+---------+------------+------------+-----+
|          0|        0|           0|           0|    0|
+-----------+---------+------------+------------+-----+



In [ ]:

print("Conso negatives:")
df_conso_numeric.filter(F.col("conso_clean") < 0).groupBy("type_energie").count().show()

print("\nConso > 10000:")
df_conso_numeric.filter(F.col("conso_clean") > 1000).groupBy("type_energie").count().show()

Conso negatives:
+------------+-----+
|type_energie|count|
+------------+-----+
|         eau|12945|
|         gaz|12997|
| electricite|12968|
+------------+-----+


Conso > 10000:
+------------+------+
|type_energie| count|
+------------+------+
|         gaz|311750|
|         eau| 12810|
| electricite|182477|
+------------+------+



### synthese

In [22]:
# problemes
total = df_raw.count()

non_numeric = df_raw.filter(
    ~F.col("consommation").rlike("^-?[0-9]+[.,]?[0-9]*$")
).count()

with_comma = df_raw.filter(F.col("consommation").contains(",")).count()

negative = df_conso_numeric.filter(F.col("conso_clean") < 0).count()

outliers = df_conso_numeric.filter(F.col("conso_clean") > 10000).count()

duplicates = total - df_raw.dropDuplicates(["batiment_id", "timestamp", "type_energie"]).count()


print(f"Total enregistrements: {total:,}")
print()
print(f"Problemes identifies:")
print(f"  - Valeurs non numeriques: {non_numeric:,} ({non_numeric/total*100:.2f}%)")
print(f"  - Valeurs avec virgule decimale: {with_comma:,} ({with_comma/total*100:.2f}%)")
print(f"  - Valeurs negatives: {negative:,} ({negative/total*100:.2f}%)")
print(f"  - Valeurs aberrantes (>1000): {outliers:,} ({outliers/total*100:.2f}%)")
print(f"  - Doublons: {duplicates:,} ({duplicates/total*100:.2f}%)")
print(f"  - Formats de dates multiples: 8 formats differents detectes, dates ambigües")

Total enregistrements: 7,758,868

Problemes identifies:
  - Valeurs non numeriques: 38,975 (0.50%)
  - Valeurs avec virgule decimale: 925,392 (11.93%)
  - Valeurs negatives: 38,910 (0.50%)
  - Valeurs aberrantes (>1000): 38,560 (0.50%)
  - Doublons: 152,134 (1.96%)
  - Formats de dates multiples: 8 formats differents detectes, dates ambigües
